# Notebook 01 — Discount Factors

**Reference:** West & Harrison, *Bayesian Forecasting and Dynamic Models* (2nd ed.), Chapter 6

**New engine function:** `kalman_filter_discount(spec, y, delta)`

## 1. Motivation: the problem with specifying $W$

In the standard DLM, the state evolution noise covariance $W$ controls how quickly the latent state is allowed to change. In practice, choosing $W$ is hard:

- It has $d(d+1)/2$ free parameters for a $d$-dimensional state.
- It interacts with $V$ in complex ways — a ratio that matters, not the individual values.
- It is unidentifiable without strong prior information or very long series.

**Discount factors** (W&H §6.3) sidestep this by replacing the full $W$ specification with a single scalar $\delta \in (0, 1]$.

## 2. The discount factor idea

Recall the predict step:

$$
R_t = G C_{t-1} G' + W_t
$$

With a discount factor $\delta$, we replace this with:

$$
\boxed{R_t = \frac{1}{\delta}\, G C_{t-1} G'}
$$

**Interpretation:**
- $\delta = 1$: no inflation — $R_t = G C_{t-1} G'$, i.e. $W = 0$ (no state evolution noise).
- $\delta < 1$: the prior covariance is *inflated* by a factor $1/\delta$ before updating. The state is allowed to drift by an amount that scales with the current posterior uncertainty.
- As $\delta \to 0$: the prior becomes completely uninformative — each observation is treated in isolation.

**Why this works:** the standard form $R_t = G C_{t-1} G' + W_t$ is equivalent to the discount form when $W_t = C_{t-1} G' G (1/\delta - 1)$ (state-dependent $W$). Discounting gives the *same predictive distribution* as this implied $W_t$, but with only one free parameter.

**Typical values:** $\delta \in [0.8, 0.99]$. Values near 0.95 are a reasonable default for slowly-varying states.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

from engine.filter import kalman_filter, kalman_filter_discount
from engine.models import make_local_level
from engine.simulate import simulate

## 3. Implementation

Open `engine/filter.py` and find `kalman_filter_discount`. The key change from the standard filter is a single line in the predict step:

```python
# Standard:
R_t = _symmetrize(G @ C_prev @ G.T + W)

# Discount:
R_t = _symmetrize(G @ C_prev @ G.T / delta)
```

Everything else — the Kalman gain, the update step, the log-likelihood — is identical to the standard filter.

In [ ]:
# Simulate a local-level series with moderate state evolution
V_true, W_true = 1.0, 0.1
spec_true = make_local_level(V=V_true, W_level=W_true)
sim = simulate(spec_true, n=200, seed=7)
y = sim.y

# Run the discount filter for several values of delta
deltas = [0.80, 0.90, 0.95, 0.99]
results = {}
for d in deltas:
    results[d] = kalman_filter_discount(spec_true, y, delta=d)

# Also run the correctly-specified standard filter for comparison
fr_std = kalman_filter(spec_true, y)

In [ ]:
fig, axes = plt.subplots(len(deltas) + 1, 1, figsize=(11, 14), sharex=True)
t = np.arange(len(y))

for ax, (delta, fr) in zip(axes, results.items()):
    std = np.sqrt(fr.C[:, 0, 0])
    ax.plot(t, y[:, 0], 'o', ms=2, alpha=0.3, color='grey')
    ax.plot(t, sim.theta_true[:, 0], 'k--', lw=1, label='true state')
    ax.plot(t, fr.m[:, 0], 'C0', lw=1.5, label=f'filtered (δ={delta})')
    ax.fill_between(t, fr.m[:, 0]-2*std, fr.m[:, 0]+2*std, alpha=0.2)
    ax.set_ylabel('y')
    ax.legend(loc='upper right', fontsize=8)

# Standard filter on the last panel
ax = axes[-1]
std = np.sqrt(fr_std.C[:, 0, 0])
ax.plot(t, y[:, 0], 'o', ms=2, alpha=0.3, color='grey')
ax.plot(t, sim.theta_true[:, 0], 'k--', lw=1, label='true state')
ax.plot(t, fr_std.m[:, 0], 'C1', lw=1.5, label='standard filter (true W)')
ax.fill_between(t, fr_std.m[:, 0]-2*std, fr_std.m[:, 0]+2*std, alpha=0.2, color='C1')
ax.set(ylabel='y', xlabel='t')
ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Discount filter — effect of δ on filtered state', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Log-likelihood as a function of delta
delta_grid = np.linspace(0.70, 1.00, 61)
logliks = [kalman_filter_discount(spec_true, y, delta=d).loglik for d in delta_grid]

best_idx = int(np.argmax(logliks))
print(f"MLE delta: {delta_grid[best_idx]:.3f}  (true W/V ratio: {W_true/V_true:.2f})")

plt.figure(figsize=(8, 3))
plt.plot(delta_grid, logliks)
plt.axvline(delta_grid[best_idx], ls='--', c='C1', label=f'best δ = {delta_grid[best_idx]:.3f}')
plt.xlabel('δ')
plt.ylabel('log marginal likelihood')
plt.title('Discount factor selection by marginal likelihood')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Exercises

**Exercise 1.** The log-likelihood profile above maximises at a particular $\hat{\delta}$. As $W$ increases (state evolves faster), should $\hat{\delta}$ increase or decrease? Verify by re-running the cell above with `W_level=0.5`.

In [ ]:
# YOUR CODE HERE
# Hint: simulate a new series with W_level=0.5 and search over delta_grid


**Exercise 2.** Posterior interval width increases with smaller $\delta$ because $R_t$ is inflated. Confirm this numerically: compute the mean posterior standard deviation $\bar{\sigma} = \frac{1}{T} \sum_t \sqrt{C_t}$ for each $\delta \in \{0.80, 0.90, 0.95, 0.99\}$ and plot it.

In [ ]:
# YOUR CODE HERE


**Exercise 3.** Show that for a local-level model ($G = F = 1$), the one-step-ahead predictive variance satisfies the recursion

$$
Q_t = \frac{V}{1 - (1-\delta) Q_{t-1}/Q_{t-1}} \quad \text{(... not quite — derive the correct form)}
$$

Specifically, write out $Q_t$ in terms of $Q_{t-1}$, $V$, and $\delta$ by substituting the discount update into the Kalman predict step. Check your derivation numerically.

In [ ]:
# YOUR CODE HERE — derive and verify
